In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-twotd_a9/unsloth_63b4a1439d50410fb2c4614be8bd5a7f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-twotd_a9/unsloth_63b4a1439d50410fb2c4614be8bd5a7f
  Resolved https://github.com/unslothai/unsloth.git to commit 4f9c8321a2136e62fd86fe722a544afd534334a5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/qwen2.5-7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [4]:
from datasets import load_dataset

medical_prompt = """أنت مساعد طبي ذكي وموثوق باللغة العربية. دورك الأساسي هو تقديم معلومات طبية عامة، وتثقيف صحي بأسلوب علمي، دقيق، ومبسط.
أجب على السؤال الطبي التالي بدقة:

### السؤال:
{}

### الإجابة:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = medical_prompt.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = load_dataset("json", data_files="qwen_ready_medical_qa.jsonl", split="train")

dataset = dataset.map(formatting_prompts_func, batched = True,)

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [8]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,783 | Num Epochs = 1 | Total steps = 1,848
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
1,1.250665
2,1.447499
3,1.325152
4,1.248947
5,1.558405
6,1.436858
7,1.400175
8,1.108096
9,1.254854
10,1.391336


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1848/tokenizer_config.json.


In [9]:
model.save_pretrained("qwen_medical_lora")
tokenizer.save_pretrained("qwen_medical_lora")
print("DONE")

Unsloth: Restored added_tokens_decoder metadata in qwen_medical_lora/tokenizer_config.json.


DONE


In [10]:
FastLanguageModel.for_inference(model)

medical_prompt = """أنت مساعد طبي ذكي وموثوق باللغة العربية. دورك الأساسي هو تقديم معلومات طبية عامة، وتثقيف صحي بأسلوب علمي، دقيق، ومبسط.
أجب على السؤال الطبي التالي بدقة:

### السؤال:
{}

### الإجابة:
{}"""

test_question = "أنا أعاني من صداع شديد جداً منذ الصباح وزغللة في عيني، هل هذا يعني أن ضغط الدم لدي مرتفع وما هو الدواء الذي يجب أن أتناوله فوراً؟"

inputs = tokenizer(
[
    medical_prompt.format(
        test_question,
        "",
    )
], return_tensors = "pt").to("cuda")

print("جاري توليد الإجابة...\n")
print("-" * 50)

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

print(response)
print("-" * 50)

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


جاري توليد الإجابة...

--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

أنت مساعد طبي ذكي وموثوق باللغة العربية. دورك الأساسي هو تقديم معلومات طبية عامة، وتثقيف صحي بأسلوب علمي، دقيق، ومبسط.
أجب على السؤال الطبي التالي بدقة:

### السؤال:
أنا أعاني من صداع شديد جداً منذ الصباح وزغللة في عيني، هل هذا يعني أن ضغط الدم لدي مرتفع وما هو الدواء الذي يجب أن أتناوله فوراً؟

### الإجابة:
لا يمكن تحديد ذلك بدون فحص طبي سريري أولاً، فهناك أسباب عديدة لذلك، وربما يكون ارتفاع ضغط الدم أحد هذه الأسباب، ولكن لا يمكن تحديد ذلك بدون فحص طبي سريري أولاً
--------------------------------------------------


In [11]:

model.save_pretrained_gguf("qwen_medical_model", tokenizer, quantization_method = "q4_k_m")

print("DONE")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in qwen_medical_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [18:21<00:00, 275.42s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [08:58<00:00, 134.63s/it]


Unsloth: Merge process complete. Saved to `/content/qwen_medical_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen_medical_model_gguf/qwen2.5-7b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['qwen_medical_model_gguf/qwen2.5-7b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/qwen2.5-7b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model qwen_medical_model_gguf/qwen2.5-7b.Q4_K_M.gguf -p "why is the sky blue?"
DONE
